# Optimized YOLOv8 OBB Training Pipeline

This notebook implements an optimized training pipeline for YOLOv8 Oriented Bounding Box (OBB) detection on an RTX 4090.

**Features:**
- Uses `ultralytics` library.
- Optimized for 24GB VRAM (Batch 64, AMP, Cache).
- Automated validation and metrics reporting.

In [ ]:
import torch
from ultralytics import YOLO
import os

# Check CUDA availability
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================

# Path to the merged dataset YAML (generated by merge_datasets.py)
DATASET_YAML = "mergeRM/data.yaml"

# Model selection (using YOLOv8n-obb as base, can upgrade to s/m/l/x)
MODEL_NAME = "yolov8n-obb.pt" 

# Project and run name for saving results
PROJECT_NAME = "robomaster_training"
RUN_NAME = "optimized_4090_run"

# Training Hyperparameters optimized for RTX 4090 (24GB VRAM)
EPOCHS = 300
IMG_SIZE = 640
BATCH_SIZE = 64  # High batch size for 4090 utilization (adjust if OOM)
WORKERS = 8      # Number of dataloader workers

## Training Pipeline

In [ ]:
def train_pipeline():
    print(f"Starting optimized training on {torch.cuda.get_device_name(0)}...")
    
    # 1. Load the model
    # Use 'yolov8n-obb.pt' for Oriented Bounding Box tasks
    model = YOLO(MODEL_NAME) 

    # 2. Train the model
    # Optimized settings:
    # - amp=True: Automatic Mixed Precision (faster on Tensor Cores)
    # - cache=True: Cache images in RAM for speed (if ample RAM available)
    # - device=0: Use the first GPU
    results = model.train(
        data=DATASET_YAML,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=0,
        workers=WORKERS,
        project=PROJECT_NAME,
        name=RUN_NAME,
        amp=True,       # Enable Mixed Precision
        cache=True,     # Cache images for faster training
        exist_ok=True,  # Overwrite existing run if name matches
        patience=50,    # Early stopping if no improvement for 50 epochs
        cos_lr=True,    # Cosine learning rate scheduler
        task='obb'      # Explicitly specify OBB task
    )
    
    return model

# Run Training
model = train_pipeline()

## Validation & Metrics
Loads the best model from the training run and computes final validation metrics.

In [ ]:
print("\n" + "="*50)
print("Running Validation on Best Model...")
print("="*50)

# Load the best model from the training run
best_model_path = os.path.join(PROJECT_NAME, RUN_NAME, "weights", "best.pt")
best_model = YOLO(best_model_path)

# Run validation
metrics = best_model.val(data=DATASET_YAML, split='val', device=0)

# Display Metrics
print("\n" + "="*50)
print("Final Validation Metrics")
print("="*50)
print(f"mAP50:       {metrics.box.map50:.4f}")
print(f"mAP50-95:    {metrics.box.map:.4f}")
print(f"Precision:   {metrics.box.mp:.4f}")
print(f"Recall:      {metrics.box.mr:.4f}")